In [ ]:
"""
L0 Evaluation — Raw Code Only (No Analysis Context)
Runs on Google Colab with HuggingFace Transformers (Qwen2.5-14B-Instruct, free T4/A100 GPU).
Split into notebook cells at the '# %%' markers.

Paper: research_paper_q1.md
Evaluation metrics:
  CFR = compiles + ASan/UBSan passes + Flawfinder no longer flags known CWE + no new CWEs
  PIR = compiles + runs but known CWE persists OR new CWEs appear
  IFR = empty / identical / won't compile / crashes at runtime / deletes function

Dataset: juliet_c_cpp_selected/ (9 CWE dirs, 100 total .c/.cpp files)
Harness:  datasets/C/testcasesupport/ (io.c, std_testcase.h, ...)
"""

"\nL0 Evaluation — Raw Code Only (No Analysis Context)\nRuns on Google Colab with HuggingFace Transformers (Qwen2.5-14B-Instruct, free T4/A100 GPU).\nSplit into notebook cells at the '# %%' markers.\n\nPaper: research_paper_q1.md\nEvaluation metrics:\n  CFR = compiles + ASan/UBSan passes + Flawfinder no longer flags known CWE + no new CWEs\n  PIR = compiles + runs but known CWE persists OR new CWEs appear\n  IFR = empty / identical / won't compile / crashes at runtime / deletes function\n\nDataset: juliet_c_cpp_selected/ (9 CWE dirs, 100 total .c/.cpp files)\nHarness:  datasets/C/testcasesupport/ (io.c, std_testcase.h, ...)\n"

In [ ]:
# !pip install transformers accelerate bitsandbytes flawfinder -q
# !apt-get update -qq && apt-get install -y -qq gcc g++ > /dev/null 2>&1

import subprocess as _sp
import sys

print("Installing Python dependencies...")
_sp.run([sys.executable, "-m", "pip", "install",
         "transformers", "accelerate", "bitsandbytes", "flawfinder", "-q"],
        check=True)

print("Installing gcc/g++...")
_sp.run(["apt-get", "update", "-qq"], check=True)
_sp.run(["apt-get", "install", "-y", "-qq", "gcc", "g++"], check=True)

Installing Python dependencies...
Installing gcc/g++...


CompletedProcess(args=['apt-get', 'install', '-y', '-qq', 'gcc', 'g++'], returncode=0)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# import os
# import re
# import json
# import time
# import subprocess
# import tempfile
# from pathlib import Path
# from datetime import datetime, timezone

# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM

import os
import re
import json
import time
import shutil
import subprocess
import tempfile
from pathlib import Path
from datetime import datetime, timezone

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Verify flawfinder is on PATH (auto-install if somehow missing)
if not shutil.which("flawfinder"):
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "flawfinder", "-q"], check=True)
    print("flawfinder installed.")

# ── CONFIGURE THESE (adjust if your Drive layout differs) ──────────────────
# Root of the cloned/copied repo on Drive:
REPO_ROOT    = Path("/content/drive/MyDrive/")

DATASET_DIR  = REPO_ROOT / "juliet_c_cpp_selected"       # 9 CWE dirs, 100 files
HARNESS_DIR  = REPO_ROOT / "/content/drive/MyDrive/testcasesupport"  # io.c + std_testcase.h
OUTPUT_DIR   = REPO_ROOT / "l0_outputs"                  # per-case fixed files
RESULTS_DIR  = REPO_ROOT / "l0_results"                  # JSON results + progress
# ───────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if not HARNESS_DIR.exists():
    raise FileNotFoundError(
        f"\n[ERROR] HARNESS_DIR not found: {HARNESS_DIR}\n"
        "Upload datasets/C/testcasesupport/ to your Drive under cve_llm/\n"
        "Or update HARNESS_DIR to the correct path."
    )
if not (HARNESS_DIR / "std_testcase.h").exists():
    raise FileNotFoundError(
        f"[ERROR] std_testcase.h not found in {HARNESS_DIR}\n"
        "Make sure the full testcasesupport/ folder is uploaded."
    )
print(f"  Harness files OK: {list(HARNESS_DIR.iterdir())}")

MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"

# CWEs whose vulnerabilities are NOT reliably detected by ASan/UBSan at runtime
# (semantic/logic bugs). For these, evaluation falls back to Flawfinder-only.
SAST_ONLY_CWES = {
    "510",   # Trapdoor (logic-based)
    "367",   # TOC/TOU (race condition)
    "36",    # Absolute Path Traversal
    "78",    # OS Command Injection
}

# How many times to run each compiled binary (Juliet uses rand() for control flow)
NUM_HARNESS_RUNS = 3

print("Paths configured.")
print(f"  DATASET_DIR : {DATASET_DIR}")
print(f"  HARNESS_DIR : {HARNESS_DIR}")
print(f"  OUTPUT_DIR  : {OUTPUT_DIR}")
print(f"  RESULTS_DIR : {RESULTS_DIR}")

  Harness files OK: [PosixPath('/content/drive/MyDrive/testcasesupport/std_testcase.h'), PosixPath('/content/drive/MyDrive/testcasesupport/std_thread.h'), PosixPath('/content/drive/MyDrive/testcasesupport/std_thread.c'), PosixPath('/content/drive/MyDrive/testcasesupport/std_testcase_io.h'), PosixPath('/content/drive/MyDrive/testcasesupport/testcases.h'), PosixPath('/content/drive/MyDrive/testcasesupport/io.c'), PosixPath('/content/drive/MyDrive/testcasesupport/main.cpp'), PosixPath('/content/drive/MyDrive/testcasesupport/main_linux.cpp')]
Paths configured.
  DATASET_DIR : /content/drive/MyDrive/juliet_c_cpp_selected
  HARNESS_DIR : /content/drive/MyDrive/testcasesupport
  OUTPUT_DIR  : /content/drive/MyDrive/l0_outputs
  RESULTS_DIR : /content/drive/MyDrive/l0_results


In [ ]:
def extract_cwe_id(dirname: str) -> str:
    """Extract CWE number from dir name, e.g. 'CWE121_Stack_...' -> '121'."""
    m = re.match(r"CWE(\d+)", dirname)
    return m.group(1) if m else ""


# Patterns that identify Windows-only Juliet variants — these include <process.h>
# and Windows spawning APIs (w32spawnl, w32spawnvp, w32_execv etc.) that don't
# exist on Linux. Skip them so they don't pollute results with unavoidable IFRs.
_WINDOWS_ONLY_PATTERNS = [
    "w32",          # w32spawnl, w32spawnvp, w32_execv, w32_spawnvp ...
    "_winsock",
    "_win32",
]

def _is_windows_only(src_file: Path) -> bool:
    """Return True if this Juliet file is a Windows-only variant."""
    stem_lower = src_file.stem.lower()
    return any(pat in stem_lower for pat in _WINDOWS_ONLY_PATTERNS)


def load_cases(dataset_dir: Path) -> list[dict]:
    """
    Load all .c/.cpp files from each CWE subdirectory.
    Windows-only variants (process.h / w32* APIs) are skipped — they cannot
    compile on Linux/Colab and would always produce a false IFR.
    """
    cases   = []
    skipped = []
    for cwe_dir in sorted(dataset_dir.iterdir()):
        if not cwe_dir.is_dir():
            continue
        cwe_id = extract_cwe_id(cwe_dir.name)
        for src_file in sorted(cwe_dir.glob("*.c")) + sorted(cwe_dir.glob("*.cpp")):
            if _is_windows_only(src_file):
                skipped.append(src_file.name)
                continue
            cases.append({
                "case_id":  src_file.stem,
                "cwe_dir":  cwe_dir.name,
                "cwe_id":   cwe_id,
                "filepath": str(src_file),
                "filename": src_file.name,
                "source":   src_file.read_text(errors="replace"),
            })
    if skipped:
        print(f"  Skipped {len(skipped)} Windows-only variants (process.h / w32*):")
        for s in skipped:
            print(f"    - {s}")
    return cases


cases = load_cases(DATASET_DIR)
print(
    f"\nLoaded {len(cases)} cases across {len(set(c['cwe_dir'] for c in cases))} CWE dirs\n"
)
for cwe in sorted(set(c["cwe_dir"] for c in cases)):
    n   = sum(1 for c in cases if c["cwe_dir"] == cwe)
    cid = next(c["cwe_id"] for c in cases if c["cwe_dir"] == cwe)
    print(f"  {cwe} (CWE-{cid}): {n} cases")


  Skipped 13 Windows-only variants (process.h / w32*):
    - CWE36_Absolute_Path_Traversal__char_listen_socket_w32CreateFile_07.cpp
    - CWE36_Absolute_Path_Traversal__wchar_t_file_w32CreateFile_17.cpp
    - CWE36_Absolute_Path_Traversal__wchar_t_listen_socket_w32CreateFile_42.cpp
    - CWE78_OS_Command_Injection__char_connect_socket_w32_spawnvp_13.c
    - CWE78_OS_Command_Injection__char_connect_socket_w32spawnl_31.c
    - CWE78_OS_Command_Injection__char_environment_w32spawnl_15.c
    - CWE78_OS_Command_Injection__char_listen_socket_w32_spawnvp_45.c
    - CWE78_OS_Command_Injection__wchar_t_connect_socket_w32_execv_18.c
    - CWE78_OS_Command_Injection__wchar_t_connect_socket_w32spawnl_05.c
    - CWE78_OS_Command_Injection__wchar_t_console_w32spawnl_09.c
    - CWE78_OS_Command_Injection__wchar_t_environment_w32_execv_06.c
    - CWE78_OS_Command_Injection__wchar_t_file_w32_execv_13.c
    - CWE78_OS_Command_Injection__char_connect_socket_w32_execv_33.cpp

Loaded 87 cases across 9 CWE 

In [ ]:
print(f"\nLoading {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,
    quantization_config=quantization_config,
    trust_remote_code=True,
)
model.eval()
print("Model loaded.")


Loading Qwen/Qwen2.5-14B-Instruct ...


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded.


In [ ]:
def build_l0_prompt(source_code: str) -> str:
    """L0 prompt: raw source code only, no SAST output, no CPG context."""
    return (
        "### Role:\n"
        "You are a secure code repair assistant specializing in identifying and "
        "fixing software vulnerabilities in source code.\n\n"
        "### Context:\n\n"
        "#### Source Code:\n"
        f"{source_code}\n\n"
        "### Task:\n"
        "Fix the vulnerabilities in the code.\n\n"
        "Requirements:\n"
        "- Preserve the original functionality of the program.\n"
        "- Modify only the parts of the code necessary to fix the vulnerabilities.\n"
        "- Do not introduce new vulnerabilities or errors.\n"
        "- Ensure the code compiles and runs correctly.\n\n"
        "### Output:\n"
        "Return only the fully corrected source code.\n"
        "Do not include explanations, comments, or additional text."
    )


@torch.inference_mode()
def call_model(prompt: str, max_new_tokens: int = 4096) -> str | None:
    """Single-shot generation. Temperature=0 for deterministic output."""
    messages = [
        {"role": "system", "content": "You are a secure code repair assistant."},
        {"role": "user",   "content": prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    try:
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # greedy / temperature=0
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
        # Decode only the newly generated tokens (skip the prompt)
        new_tokens = out[0][inputs["input_ids"].shape[-1]:]
        return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    except Exception as e:
        print(f"  [MODEL ERROR] {e}")
        return None


def extract_code(response: str) -> str:
    """Strip markdown code fences that some models wrap output in."""
    if not response:
        return ""
    # Handle ```c / ```cpp / ``` fences
    fenced = re.search(r"```(?:c|cpp|c\+\+)?\n(.*?)```", response, re.DOTALL)
    if fenced:
        return fenced.group(1).strip()
    lines = response.strip().splitlines()
    if lines and lines[0].startswith("```"):
        lines = lines[1:]
    if lines and lines[-1].strip() == "```":
        lines = lines[:-1]
    return "\n".join(lines)


# Quick sanity-check
print("Model test:", call_model("Say hello in one word.", max_new_tokens=10))

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Model test: Hello


In [ ]:
progress_file = RESULTS_DIR / "l0_progress.json"
completed: dict[str, dict] = {}
if progress_file.exists():
    completed = json.loads(progress_file.read_text())
    print(f"Resuming — {len(completed)} cases already done.")

print(f"\n{'='*60}")
print(f"L0 generation: {len(cases)} cases | model={MODEL_ID}")
print(f"{'='*60}\n")

for i, case in enumerate(cases, 1):
    cid = case["case_id"]
    if cid in completed:
        print(f"[{i:3}/{len(cases)}] SKIP: {cid}")
        continue

    print(f"[{i:3}/{len(cases)}] CWE-{case['cwe_id']} | {cid}")

    prompt = build_l0_prompt(case["source"])
    t0     = time.time()
    raw    = call_model(prompt)
    elapsed = round(time.time() - t0, 1)

    fix    = extract_code(raw) if raw else ""
    status = "generated" if raw else "error"

    # Save fix file (mirrors dataset structure)
    out_file = OUTPUT_DIR / case["cwe_dir"] / case["filename"]
    out_file.parent.mkdir(parents=True, exist_ok=True)
    out_file.write_text(fix)

    completed[cid] = {
        "case_id":   cid,
        "cwe_dir":   case["cwe_dir"],
        "cwe_id":    case["cwe_id"],
        "filename":  case["filename"],
        "status":    status,
        "elapsed_s": elapsed,
        "fix_len":   len(fix),
    }
    progress_file.write_text(json.dumps(completed, indent=2))
    print(f"  -> [{status}] {elapsed}s | {len(fix)} chars")

print(f"\nGeneration complete: {len(completed)}/{len(cases)}")

Resuming — 6 cases already done.

L0 generation: 100 cases | model=Qwen/Qwen2.5-14B-Instruct

[  1/100] SKIP: CWE121_Stack_Based_Buffer_Overflow__CWE129_fscanf_34
[  2/100] SKIP: CWE121_Stack_Based_Buffer_Overflow__CWE131_memmove_02
[  3/100] SKIP: CWE121_Stack_Based_Buffer_Overflow__CWE193_wchar_t_alloca_ncpy_12
[  4/100] SKIP: CWE121_Stack_Based_Buffer_Overflow__CWE193_wchar_t_declare_loop_18
[  5/100] SKIP: CWE121_Stack_Based_Buffer_Overflow__CWE805_char_alloca_snprintf_17
[  6/100] SKIP: CWE121_Stack_Based_Buffer_Overflow__CWE805_int_alloca_memmove_07
[  7/100] CWE-121 | CWE121_Stack_Based_Buffer_Overflow__CWE805_int_alloca_memmove_18
  -> [generated] 8.1s | 1248 chars
[  8/100] CWE-121 | CWE121_Stack_Based_Buffer_Overflow__CWE805_wchar_t_declare_snprintf_32
  -> [generated] 11.6s | 1857 chars
[  9/100] CWE-121 | CWE121_Stack_Based_Buffer_Overflow__dest_wchar_t_alloca_cpy_15
  -> [generated] 12.3s | 1957 chars
[ 10/100] CWE-121 | CWE121_Stack_Based_Buffer_Overflow__CWE129_listen_so

In [ ]:
def _compile_and_run(
    code: str, filename: str, tmpdir: str, label: str = "test"
) -> dict:
    """
    Compile a single Juliet file with ASan+UBSan, link against io.c, and run it.
    -DINCLUDEMAIN activates the built-in main() in each Juliet test file.

    Harness files (std_testcase.h, io.c, etc.) are copied into tmpdir before
    compilation so the -I flag always resolves regardless of HARNESS_DIR path.

    Returns: {compiles, runs, compile_err, run_err}
    """
    import shutil
    ext = ".cpp" if filename.endswith(".cpp") else ".c"
    cc = "g++" if ext == ".cpp" else "gcc"

    # Copy ALL harness support files into tmpdir so #include "std_testcase.h" resolves
    for hfile in HARNESS_DIR.iterdir():
        if hfile.is_file():
            shutil.copy2(str(hfile), os.path.join(tmpdir, hfile.name))

    io_src   = os.path.join(tmpdir, "io.c")   # now lives in tmpdir
    src_path = os.path.join(tmpdir, f"{label}_{filename}")
    bin_path = os.path.join(tmpdir, f"{label}_bin")

    with open(src_path, "w") as f:
        f.write(code)

    compile_cmd = [
        cc,
        "-DINCLUDEMAIN",
        "-fsanitize=address,undefined",
        "-fno-sanitize-recover=all",
        "-I", tmpdir,            # headers now live here
        src_path,
        io_src,
        "-o", bin_path,
        "-lm",
        "-lpthread",             # needed by socket/thread Juliet cases
    ]

    try:
        comp = subprocess.run(compile_cmd, capture_output=True, text=True, timeout=30)
    except subprocess.TimeoutExpired:
        return {
            "compiles": False,
            "runs": False,
            "compile_err": "Compile timed out",
            "run_err": "",
        }
    except Exception as e:
        return {"compiles": False, "runs": False, "compile_err": str(e), "run_err": ""}

    if comp.returncode != 0:
        err = comp.stderr[:1500]
        # Detect the specific case where the model removed the #ifdef INCLUDEMAIN
        # wrapper — the linker then can't find main() and fails with this message.
        if "undefined reference to `main'" in comp.stderr:
            err = "Model removed #ifdef INCLUDEMAIN block (no main() compiled in)"
        return {
            "compiles": False,
            "runs":     False,
            "compile_err": err,
            "run_err":  "",
        }

    # Run multiple times — Juliet uses rand()/srand(time()) for control-flow
    env = {
        **os.environ,
        "ASAN_OPTIONS": "detect_leaks=0",
        "UBSAN_OPTIONS": "print_stacktrace=1",
    }
    for run_idx in range(NUM_HARNESS_RUNS):
        try:
            run = subprocess.run(
                [bin_path],
                capture_output=True,
                text=True,
                timeout=10,
                env=env,
            )
        except subprocess.TimeoutExpired:
            return {
                "compiles": True,
                "runs": False,
                "compile_err": "",
                "run_err": f"Runtime timed out (run {run_idx + 1}/{NUM_HARNESS_RUNS})",
            }
        except Exception as e:
            return {
                "compiles": True,
                "runs": False,
                "compile_err": "",
                "run_err": str(e),
            }

        if run.returncode != 0:
            err = run.stderr[:500] if run.stderr else f"Exit code {run.returncode}"
            return {"compiles": True, "runs": False, "compile_err": "", "run_err": err}

    return {"compiles": True, "runs": True, "compile_err": "", "run_err": ""}


def juliet_harness_test(original: str, fix: str, filename: str, cwe_id: str) -> dict:
    """
    Two-step harness:
      1. Compile + run the ORIGINAL under ASan/UBSan — expect a crash
         (confirms the sanitiser fires on the vulnerable code).
         For semantic CWEs (SAST_ONLY_CWES), skip runtime check — harness unreliable.
      2. Compile + run the FIX — expect clean exit.

    Returns:
      baseline_crashes  : original triggered sanitiser (True = harness is reliable)
      harness_reliable  : whether runtime result is meaningful for this CWE
      fix_compiles      : bool
      fix_runs          : bool
      compile_err       : str
      run_err           : str
    """
    if not fix.strip():
        return {
            "baseline_crashes": False,
            "harness_reliable": False,
            "fix_compiles": False,
            "fix_runs": False,
            "compile_err": "Empty source",
            "run_err": "",
        }

    runtime_meaningful = cwe_id not in SAST_ONLY_CWES

    with tempfile.TemporaryDirectory() as tmpdir:
        baseline_crashes = False
        if runtime_meaningful:
            orig = _compile_and_run(original, filename, tmpdir, label="orig")
            # crash = compiles but non-zero exit (sanitiser abort)
            baseline_crashes = orig["compiles"] and not orig["runs"]

        fix_result = _compile_and_run(fix, filename, tmpdir, label="fix")

        return {
            "baseline_crashes": baseline_crashes,
            "harness_reliable": runtime_meaningful,
            "fix_compiles": fix_result["compiles"],
            "fix_runs": fix_result["runs"]
            if runtime_meaningful
            else fix_result["compiles"],
            "compile_err": fix_result["compile_err"],
            "run_err": fix_result["run_err"],
        }


def run_flawfinder(code: str, filename: str) -> list[dict]:
    """Run Flawfinder on code, return list of findings with CWE IDs."""
    if not code.strip():
        return []
    ext = ".cpp" if filename.endswith(".cpp") else ".c"
    with tempfile.NamedTemporaryFile(mode="w", suffix=ext, delete=False) as f:
        f.write(code)
        tmp = f.name
    try:
        r = subprocess.run(
            ["flawfinder", "--csv", "--quiet", tmp],
            capture_output=True,
            text=True,
            timeout=60,
        )
        findings = []
        lines = r.stdout.strip().splitlines()
        for line in lines[1:]:  # skip CSV header
            parts = line.split(",")
            if len(parts) >= 5:
                cwe_field = parts[-1].strip().strip('"')
                cwe_nums = re.findall(r"CWE-(\d+)", cwe_field)
                findings.append(
                    {
                        "line": parts[1].strip().strip('"'),
                        "level": parts[3].strip().strip('"'),
                        "category": parts[4].strip().strip('"')
                        if len(parts) > 4
                        else "",
                        "name": parts[5].strip().strip('"') if len(parts) > 5 else "",
                        "cwes": cwe_nums,
                        "raw": line,
                    }
                )
        return findings
    except Exception as e:
        print(f"  [Flawfinder error] {e}")
        return []
    finally:
        os.unlink(tmp)


def function_deleted(original: str, fix: str) -> bool:
    """Heuristic: fix is less than 30% the size of original → function was gutted."""
    return len(fix.strip()) < len(original.strip()) * 0.30

In [ ]:
def classify(original: str, fix: str, filename: str, known_cwe: str) -> dict:
    """
    Classify a fix per paper (research_paper_q1.md §Evaluation Metrics):

    Pre-checks (IFR):
      - Empty output
      - Identical to original
      - Gutted function body

    Stage 1 — Juliet harness (compile + ASan/UBSan runtime):
      IFR if fix won't compile or crashes at runtime.

    Stage 2 — Flawfinder SAST rescan:
      CFR if known CWE resolved AND no new CWEs introduced.
      PIR if known CWE persists OR new CWEs introduced.
    """
    # ── IFR pre-checks ──────────────────────────────────────────────────────
    if not fix.strip():
        return {
            "outcome": "IFR", "reason": "Empty output",
            "compile_ok": False, "runs_ok": False, "harness_reliable": False,
        }
    if fix.strip() == original.strip():
        return {
            "outcome": "IFR", "reason": "Identical to original",
            "compile_ok": False, "runs_ok": False, "harness_reliable": False,
        }
    if function_deleted(original, fix):
        return {
            "outcome": "IFR", "reason": "Function body deleted (< 30% original size)",
            "compile_ok": False, "runs_ok": False, "harness_reliable": False,
        }

    # ── Stage 1: Juliet harness ──────────────────────────────────────────────
    harness = juliet_harness_test(original, fix, filename, known_cwe)

    if not harness["fix_compiles"]:
        return {
            "outcome": "IFR",
            "reason": f"Compile failed: {harness['compile_err'][:200]}",
            "compile_ok": False, "runs_ok": False,
            "harness_reliable": harness["harness_reliable"],
        }

    if harness["harness_reliable"] and not harness["fix_runs"]:
        return {
            "outcome": "IFR",
            "reason": f"Runtime crash (ASan/UBSan): {harness['run_err'][:200]}",
            "compile_ok": True, "runs_ok": False,
            "harness_reliable": True,
            "baseline_crashes": harness["baseline_crashes"],
        }

    # ── Stage 2: Flawfinder SAST rescan ─────────────────────────────────────
    orig_findings = run_flawfinder(original, filename)
    fix_findings  = run_flawfinder(fix, filename)

    orig_cwes = set()
    for f in orig_findings:
        orig_cwes.update(f["cwes"])

    fix_cwes = set()
    for f in fix_findings:
        fix_cwes.update(f["cwes"])

    # Known CWE resolved if: it appeared in original AND is gone in fix
    # If it never appeared in original (Flawfinder missed it), count as resolved
    known_resolved = (known_cwe not in fix_cwes) if (known_cwe in orig_cwes) else True
    new_cwes       = fix_cwes - orig_cwes

    if known_resolved and not new_cwes:
        outcome = "CFR"
        reason  = "Harness pass + known CWE resolved + no new CWEs"
    elif new_cwes:
        outcome = "PIR"
        reason  = f"New CWEs introduced: {sorted(new_cwes)}"
    else:
        outcome = "PIR"
        reason  = f"Known CWE-{known_cwe} still flagged by Flawfinder"

    return {
        "outcome":              outcome,
        "reason":               reason,
        "compile_ok":           True,
        "runs_ok":              harness["fix_runs"],
        "harness_reliable":     harness["harness_reliable"],
        "baseline_crashes":     harness.get("baseline_crashes", False),
        "known_cwe":            known_cwe,
        "known_resolved":       known_resolved,
        "orig_cwes":            sorted(orig_cwes),
        "fix_cwes":             sorted(fix_cwes),
        "new_cwes":             sorted(new_cwes),
        "orig_findings_count":  len(orig_findings),
        "fix_findings_count":   len(fix_findings),
    }

In [ ]:
results = []
counts  = {"CFR": 0, "PIR": 0, "IFR": 0}

print(f"Evaluating {len(cases)} fixes...\n")

for i, case in enumerate(cases, 1):
    fix_path = OUTPUT_DIR / case["cwe_dir"] / case["filename"]

    if not fix_path.exists():
        cl = {
            "outcome": "IFR", "reason": "No output file generated",
            "compile_ok": False, "runs_ok": False,
        }
    else:
        fix_src = fix_path.read_text(errors="replace")
        cl = classify(case["source"], fix_src, case["filename"], case["cwe_id"])

    counts[cl["outcome"]] += 1
    results.append({
        "case_id":  case["case_id"],
        "cwe_dir":  case["cwe_dir"],
        "cwe_id":   case["cwe_id"],
        "filename": case["filename"],
        **cl,
    })
    print(f"[{i:3}/{len(cases)}] [{cl['outcome']}] {case['case_id']}  —  {cl['reason']}")

total = len(results)
print(f"\n{'='*60}")
print(f"L0 Results — {total} cases | model={MODEL_ID}")
print(f"{'='*60}")
print(f"  CFR (Correct Fix Rate):         {counts['CFR']:3}  ({100*counts['CFR']/total:.1f}%)")
print(f"  PIR (Plausible but Incorrect):  {counts['PIR']:3}  ({100*counts['PIR']/total:.1f}%)")
print(f"  IFR (Invalid Fix Rate):         {counts['IFR']:3}  ({100*counts['IFR']/total:.1f}%)")

Evaluating 87 fixes...

[  1/87] [CFR] CWE121_Stack_Based_Buffer_Overflow__CWE129_fscanf_34  —  Harness pass + known CWE resolved + no new CWEs
[  2/87] [IFR] CWE121_Stack_Based_Buffer_Overflow__CWE131_memmove_02  —  Compile failed: Model removed #ifdef INCLUDEMAIN block (no main() compiled in)
[  3/87] [IFR] CWE121_Stack_Based_Buffer_Overflow__CWE193_wchar_t_alloca_ncpy_12  —  Compile failed: Model removed #ifdef INCLUDEMAIN block (no main() compiled in)
[  4/87] [IFR] CWE121_Stack_Based_Buffer_Overflow__CWE193_wchar_t_declare_loop_18  —  Compile failed: Model removed #ifdef INCLUDEMAIN block (no main() compiled in)
[  5/87] [CFR] CWE121_Stack_Based_Buffer_Overflow__CWE805_char_alloca_snprintf_17  —  Harness pass + known CWE resolved + no new CWEs
[  6/87] [IFR] CWE121_Stack_Based_Buffer_Overflow__CWE805_int_alloca_memmove_07  —  Compile failed: /tmp/tmptf2zwb0v/fix_CWE121_Stack_Based_Buffer_Overflow__CWE805_int_alloca_memmove_07.c: In function ‘main’:
/tmp/tmptf2zwb0v/fix_CWE121_Stac

In [ ]:
final = {
    "level":       "L0",
    "model":       MODEL_ID,
    "prompt_type": "raw_code_only",
    "timestamp":   datetime.now(timezone.utc).isoformat(),
    "total_cases": total,
    "summary": {
        "CFR":     counts["CFR"],
        "PIR":     counts["PIR"],
        "IFR":     counts["IFR"],
        "CFR_pct": round(100 * counts["CFR"] / total, 1),
        "PIR_pct": round(100 * counts["PIR"] / total, 1),
        "IFR_pct": round(100 * counts["IFR"] / total, 1),
    },
    "cases": results,
}

results_file = RESULTS_DIR / "l0_results.json"
results_file.write_text(json.dumps(final, indent=2, default=str))
print(f"Results saved -> {results_file}")

# Per-CWE breakdown table
print("\nPer-CWE Breakdown:")
print(f"{'CWE Directory':<50} {'N':>3} {'CFR':>4} {'PIR':>4} {'IFR':>4}")
print("-" * 70)
for cwe in sorted(set(r["cwe_dir"] for r in results)):
    cwe_r = [r for r in results if r["cwe_dir"] == cwe]
    n   = len(cwe_r)
    cfr = sum(1 for r in cwe_r if r["outcome"] == "CFR")
    pir = sum(1 for r in cwe_r if r["outcome"] == "PIR")
    ifr = sum(1 for r in cwe_r if r["outcome"] == "IFR")
    print(f"  {cwe:<48} {n:>3} {cfr:>4} {pir:>4} {ifr:>4}")

# ── L1 failure pool (non-CFR cases escalate to next context level) ──
l1_pool = [r for r in results if r["outcome"] != "CFR"]
l1_file = RESULTS_DIR / "l1_failure_pool.json"
l1_file.write_text(json.dumps(l1_pool, indent=2, default=str))
print(f"\nL1 failure pool: {len(l1_pool)} cases  ->  {l1_file}")

Results saved -> /content/drive/MyDrive/l0_results/l0_results.json

Per-CWE Breakdown:
CWE Directory                                        N  CFR  PIR  IFR
----------------------------------------------------------------------
  CWE121_Stack_Based_Buffer_Overflow                10    3    0    7
  CWE122_Heap_Based_Buffer_Overflow                  8    5    0    3
  CWE190_Integer_Overflow                           10    5    0    5
  CWE367_TOC_TOU                                    16    3    0   13
  CWE36_Absolute_Path_Traversal                      7    1    0    6
  CWE415_Double_Free                                10    8    0    2
  CWE416_Use_After_Free                             14   13    0    1
  CWE510_Trapdoor                                   10    8    0    2
  CWE78_OS_Command_Injection                         2    0    0    2

L1 failure pool: 41 cases  ->  /content/drive/MyDrive/l0_results/l1_failure_pool.json
